# Análisis espacio-temporal del cáncer infantil en Colombia, 2020–2024

**Grupo GEMMA — Universidad de Sucre**
Nohelis Muslaco Bohorquez · Maria Mendez R. · Melba Vertel M.

3.ª Feria de Innovación, Universidad de Córdoba (CONASIE 2026)

---

Este cuaderno rehace el análisis completo y reproduce, cifra por cifra, los resultados
obtenidos en R con `MASS::glm.nb` y `performance::model_performance`.

| Sección | Contenido |
|---|---|
| 1 | Preparación y carga de datos |
| 2 | De la base completa a `DataE` |
| 3 | Exploración: cómo se reparten los casos |
| 4 | Modelo binomial negativa |
| 5 | Razones de tasas (IRR) |
| 6 | Índices de desempeño del modelo |
| 7 | Por qué no una regresión lineal |
| 8 | Resumen y exportación |

> **Para ejecutarlo:** *Entorno de ejecución → Ejecutar todo*. No hace falta subir nada:
> el cuaderno descarga los datos del repositorio.

## 1. Preparación

### 1.1 Entorno y datos

Si el cuaderno se abre desde Colab con el botón del repositorio, la carpeta ya está disponible. Si no, esta celda la clona.

In [ ]:
import os, subprocess, json, warnings
warnings.filterwarnings("ignore")

# Repositorio del proyecto:
REPO_URL = "https://github.com/MariaClareth/Monitoreo-del-cancer-infantil-en-Colombia"

def carpeta_del_proyecto():
    for c in [".", "..", "/content/Monitoreo-del-cancer-infantil-en-Colombia"]:
        if os.path.exists(os.path.join(c, "datos", "DataE.csv")):
            return os.path.abspath(c)
    subprocess.run(["git", "clone", "-q", REPO_URL + ".git",
                    "/content/Monitoreo-del-cancer-infantil-en-Colombia"], check=True)
    return "/content/Monitoreo-del-cancer-infantil-en-Colombia"

os.chdir(carpeta_del_proyecto())
print("Carpeta de trabajo:", os.getcwd())
print("Archivos de datos :", os.listdir("datos"))

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats as st
import matplotlib
import matplotlib.pyplot as plt

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 30)
np.random.seed(2024)

print("pandas", pd.__version__, "| statsmodels", sm.__version__)

### 1.2 Estilo de las figuras

Se conserva el aspecto de `ggplot2` para que las figuras de este cuaderno se vean iguales a las del póster.

In [ ]:
AZUL_CAJA = "#7B68EE"     # el mismo relleno de los boxplots originales
TINTA, APAGADO = "#233043", "#5B6C82"
NARANJA, NAVY = "#E8552D", "#12336B"

def panel_ggplot(ax):
    "Reproduce theme_gray() de ggplot2: panel gris con rejilla blanca."
    ax.set_facecolor("#EBEBEB")
    ax.grid(True, color="white", linewidth=1.1)
    ax.set_axisbelow(True)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(length=0, colors="#4D4D4D", labelsize=11)

plt.rcParams.update({
    "font.family": "DejaVu Sans", "figure.dpi": 110, "savefig.dpi": 300,
    "figure.facecolor": "white", "savefig.facecolor": "white",
    "axes.labelcolor": "#1A1A1A", "text.color": "#1A1A1A", "axes.labelsize": 13,
})
os.makedirs("figuras", exist_ok=True)
os.makedirs("resultados", exist_ok=True)

## 2. De la base completa a `DataE`

La base tiene una fila por notificación. Los modelos trabajan sobre conteos, así que se
agrupa por las cuatro variables de interés y se cuenta.

Dos decisiones importantes:

- **Las combinaciones sin casos se conservan como cero.** Si se omitieran, el modelo vería
  solo las celdas con casos y sobreestimaría los conteos de las zonas pequeñas.
- **Se excluye la zona insular** (San Andrés y Providencia, 27 notificaciones): con esos
  conteos la estimación de un efecto propio de zona no es estable.

In [ ]:
base = pd.read_csv("datos/base_completa.csv")
print("Base completa:", base.shape)

ZONAS = ["AMAZ", "AND", "CAR", "ORIN", "PACF"]
d = base[base.ZONA.isin(ZONAS)].copy()
print(f"Notificaciones con zona analizable: {len(d):,}".replace(",", "."))
print("Excluidas — zona insular:", int((base.ZONA == 'INS').sum()),
      "| sin zona:", int(base.ZONA.isna().sum()))

rejilla = pd.MultiIndex.from_product(
    [ZONAS, sorted(d.ANO.unique()), sorted(d.GRUP_ETARIOS.unique()), sorted(d.SEXO.unique())],
    names=["zona", "anno", "edad", "sexo"])

DataE = (d.groupby(["ZONA", "ANO", "GRUP_ETARIOS", "SEXO"]).size()
           .rename("casos").reindex(rejilla, fill_value=0).reset_index())

print(f"\nDataE: {DataE.shape[0]} registros · {DataE.casos.sum():,} casos".replace(",", "."))
print("Celdas en cero:", int((DataE.casos == 0).sum()))
DataE.head(8)

In [ ]:
# Comprobación: el archivo del repositorio y el que acabamos de construir coinciden
DataE_repo = pd.read_csv("datos/DataE.csv")
assert DataE.equals(DataE_repo), "La reconstrucción no coincide con datos/DataE.csv"
print("La reconstrucción coincide exactamente con datos/DataE.csv")

print("\nCasos por zona:")
print(DataE.groupby("zona").casos.sum().sort_values(ascending=False).to_string())
print("\nCasos por año:")
print(DataE.groupby("anno").casos.sum().to_string())

## 3. Exploración

### Figura 1 · Casos por año y por zona

La barra azul marca la media de cada grupo. El contraste entre los dos paneles ya anticipa el resultado del modelo: por año las medias son casi idénticas; por zona, no.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, var, etiqueta in zip(axes, ["anno", "zona"], ["anno", "zona"]):
    cats = sorted(DataE[var].unique())
    pos = {c: i for i, c in enumerate(cats)}
    x = DataE[var].map(pos) + np.random.uniform(-0.13, 0.13, len(DataE))
    ax.scatter(x, DataE.casos, s=45, color="#1A1A1A", alpha=0.35,
               edgecolor="#4D4D4D", linewidth=0.4, zorder=3)
    for c in cats:
        m = DataE.loc[DataE[var] == c, "casos"].mean()
        ax.plot([pos[c]-0.34, pos[c]+0.34], [m, m], color="blue", lw=5, zorder=4)
    ax.set_xticks(range(len(cats))); ax.set_xticklabels(cats)
    ax.set_xlabel(etiqueta); ax.set_ylabel("casos  (mean)")
    ax.set_xlim(-0.6, len(cats)-0.4)
    panel_ggplot(ax)
fig.tight_layout()
fig.savefig("figuras/fig1_medias_anno_zona.png", bbox_inches="tight")
plt.show()

### Figura 2 · Casos por año, dentro de cada zona

Cada panel es una zona. Dentro de cada una, la distribución apenas se mueve de un año a otro.

In [ ]:
def cajas_por_facetas(datos, x, faceta, archivo, orden_x=None, orden_f=None, alto=2.0):
    fx = orden_x or sorted(datos[x].unique())
    ff = orden_f or sorted(datos[faceta].unique())
    fig, axes = plt.subplots(len(ff), 1, figsize=(11, alto*len(ff)), sharex=True, sharey=True)
    axes = np.atleast_1d(axes)
    ymax = datos.casos.max()*1.08
    for ax, f in zip(axes, ff):
        sub = datos[datos[faceta] == f]
        grupos = [sub.loc[sub[x] == c, "casos"].values for c in fx]
        bp = ax.boxplot(grupos, positions=range(len(fx)), widths=0.62, patch_artist=True,
                        medianprops=dict(color="black", lw=2.4),
                        boxprops=dict(facecolor=AZUL_CAJA, edgecolor="black", lw=1.1),
                        whiskerprops=dict(color="black", lw=1.4),
                        capprops=dict(color="black", lw=0),
                        flierprops=dict(marker="o", markersize=6, markerfacecolor="#808080",
                                        markeredgecolor="#4D4D4D", alpha=0.75))
        ax.set_ylim(-4, ymax)
        panel_ggplot(ax)
        ax.text(1.012, 0.5, str(f), transform=ax.transAxes, rotation=270,
                va="center", ha="left", fontsize=12, color="#1A1A1A",
                bbox=dict(boxstyle="square,pad=0.45", fc="#D5D5D5", ec="none"))
    axes[-1].set_xticks(range(len(fx))); axes[-1].set_xticklabels(fx)
    axes[-1].set_xlabel(x)
    fig.text(0.045, 0.5, "casos", rotation=90, va="center", fontsize=13)
    fig.subplots_adjust(hspace=0.12, left=0.1, right=0.94)
    fig.savefig(archivo, bbox_inches="tight")
    plt.show()

cajas_por_facetas(DataE, "anno", "zona", "figuras/fig2_anno_por_zona.png")

### Figura 3 · Casos por zona, dentro de cada año

La misma información girada. Aquí se ve que el orden de las zonas se repite igual los cinco años: la Andina siempre arriba, la Amazonía siempre abajo.

In [ ]:
cajas_por_facetas(DataE, "zona", "anno", "figuras/fig3_zona_por_anno.png")

### Figura 4 · Casos por grupo de edad y sexo

In [ ]:
cajas_por_facetas(DataE, "edad", "sexo", "figuras/fig4_edad_por_sexo.png",
                  orden_x=["e<1", "e1-5", "e14-18", "e5-9", "e9-14"], alto=3.2)

## 4. Modelo binomial negativa

$$\log(\mu_i)=\beta_0+\beta_{\text{zona}(i)}+\beta_{\text{año}}\cdot\text{año}_i,
\qquad Y_i\sim\text{BN}(\mu_i,\theta)$$

La binomial negativa permite que la varianza supere a la media, que es lo que ocurre con
estos conteos. La categoría de referencia es la **Amazonía**, así que cada coeficiente de
zona se lee como comparación contra ella.

`theta` se toma del ajuste original en R ($\hat\theta = 3{,}474$), de modo que los
coeficientes, errores estándar y desviaciones sean los mismos.

In [ ]:
THETA = 3.473998965                    # estimado por glm.nb en R
familia_bn = sm.families.NegativeBinomial(alpha=1/THETA)

m_bn = smf.glm("casos ~ C(zona, Treatment('AMAZ')) + anno",
               data=DataE, family=familia_bn).fit()

nombres = ["(Intercept)", "zonaAND", "zonaCAR", "zonaORIN", "zonaPACF", "anno"]
tabla = pd.DataFrame({
    "Estimate":  m_bn.params.values,
    "Std. Error": m_bn.bse.values,
    "z value":   m_bn.tvalues.values,
    "Pr(>|z|)":  m_bn.pvalues.values,
}, index=nombres)

def estrellas(p):
    return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "." if p < .1 else " "
tabla["signif"] = [estrellas(p) for p in tabla["Pr(>|z|)"]]

print("Coefficients:")
print(tabla.to_string(float_format=lambda v: f"{v:11.5f}"))
print("\n---")
print("Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1")
print(f"\n(Dispersion parameter for Negative Binomial({THETA:.3f}) family taken to be 1)")
print(f"\n    Null deviance: {m_bn.null_deviance:7.2f}  on {int(m_bn.df_model + m_bn.df_resid)} degrees of freedom")
print(f"Residual deviance: {m_bn.deviance:7.2f}  on {int(m_bn.df_resid)} degrees of freedom")
print(f"\n              Theta:  {THETA:.3f}")

Comparación con la salida de R, para dejar constancia de que es el mismo ajuste:

| Coeficiente | R | Este cuaderno |
|---|---|---|
| zonaAND | 3,22845 | ver arriba |
| zonaCAR | 1,90346 | |
| zonaORIN | 0,74212 | |
| zonaPACF | 1,89662 | |
| anno | 0,02614 | |
| Desviación nula / residual | 1054,10 / 272,59 | |

In [ ]:
# Verificación automática contra los valores de R
esperado = {"zonaAND": 3.22845, "zonaCAR": 1.90346, "zonaORIN": 0.74212,
            "zonaPACF": 1.89662, "anno": 0.02614}
for k, v in esperado.items():
    obtenido = tabla.loc[k, "Estimate"]
    assert abs(obtenido - v) < 1e-4, f"{k}: {obtenido:.5f} != {v}"
assert abs(m_bn.null_deviance - 1054.10) < 0.02
assert abs(m_bn.deviance - 272.59) < 0.02
print("Todos los coeficientes y las desviaciones coinciden con la salida de R.")

## 5. Razones de tasas

Exponenciar cada coeficiente lo convierte en una **razón de tasas de notificación (IRR)**:
cuántas veces más casos registra esa zona frente a la Amazonía, con el año fijo.

In [ ]:
NOMBRE_ZONA = {"zonaAND": "Andina", "zonaCAR": "Caribe",
               "zonaORIN": "Orinoquía", "zonaPACF": "Pacífico"}

ic = m_bn.conf_int()
irr = pd.DataFrame({
    "IRR": np.exp(m_bn.params.values),
    "IC 95% inf": np.exp(ic[0].values),
    "IC 95% sup": np.exp(ic[1].values),
    "p": m_bn.pvalues.values,
}, index=nombres)

zonas_irr = irr.loc[list(NOMBRE_ZONA)].rename(index=NOMBRE_ZONA)
zonas_irr.index.name = "frente a la Amazonía"
display(zonas_irr.round(3))

print(f"\nEfecto del año: IRR = {irr.loc['anno','IRR']:.4f} "
      f"(IC 95 % {irr.loc['anno','IC 95% inf']:.3f}–{irr.loc['anno','IC 95% sup']:.3f}), "
      f"p = {irr.loc['anno','p']:.3f}")
print("El intervalo contiene el 1: no hay evidencia de cambio año a año.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
z = zonas_irr.sort_values("IRR")
y = np.arange(len(z))
ax.hlines(y, z["IC 95% inf"], z["IC 95% sup"], color="#9FB6C9", lw=3)
ax.scatter(z["IRR"], y, s=160, color=NARANJA, edgecolor="white", linewidth=1.6, zorder=3)
ax.axvline(1, color=APAGADO, lw=1.4, ls=(0, (5, 4)))
for i, (nm, r) in enumerate(z.iterrows()):
    ax.text(r["IC 95% sup"]*1.06, i, f"{r['IRR']:.1f} ×", va="center", fontsize=12, color=TINTA)
ax.set_yticks(y); ax.set_yticklabels(z.index, fontsize=12)
ax.set_xscale("log"); ax.set_xlim(0.8, 60)
ax.set_xticks([1, 2, 5, 10, 25, 50]); ax.set_xticklabels(["1", "2", "5", "10", "25", "50"])
ax.set_xlabel("Razón de tasas frente a la Amazonía (escala logarítmica)")
ax.set_title("Cuántas veces más casos registra cada zona", loc="left",
             weight="bold", color=NAVY, fontsize=14, pad=12)
panel_ggplot(ax)
fig.tight_layout()
fig.savefig("figuras/fig5_irr_por_zona.png", bbox_inches="tight")
plt.show()

## 6. Índices de desempeño

Los mismos que devuelve `performance::model_performance(fit1)` en R.

In [ ]:
n = len(DataE)
k = 6 + 1                                   # 6 coeficientes + theta, como cuenta R
logL = m_bn.llf

AIC  = -2*logL + 2*k
AICc = AIC + 2*k*(k+1)/(n - k - 1)
BIC  = -2*logL + k*np.log(n)
# Nagelkerke basado en desviación, que es lo que usa performance para glm.nb
R2N  = ((1 - np.exp((m_bn.deviance - m_bn.null_deviance)/n)) /
        (1 - np.exp(-m_bn.null_deviance/n)))
RMSE = np.sqrt(np.mean((DataE.casos - m_bn.fittedvalues)**2))

desempeno = pd.DataFrame({
    "Índice": ["AIC", "AICc", "BIC", "R² de Nagelkerke", "RMSE", "Sigma",
               "Score_log", "Score_spherical"],
    "Este cuaderno": [f"{AIC:.3f}", f"{AICc:.3f}", f"{BIC:.3f}", f"{R2N:.3f}",
                      f"{RMSE:.3f}", "1.000", "—", "—"],
    "R (performance)": ["1695.667", "1696.130", "1720.318", "0.970", "15.000",
                        "1.000", "-3.389", "0.045"],
})
display(desempeno)
print("Las reglas de puntuación (Score_log y Score_spherical) se reportan tal como las")
print("entrega el paquete performance; no se recalculan aquí.")

El $R^2$ de Nagelkerke de 0,970 sale de comparar las dos desviaciones:

$$R^2_{N}=\frac{1-\exp\!\big(\tfrac{D-D_0}{n}\big)}{1-\exp\!\big(\tfrac{-D_0}{n}\big)}
=\frac{1-\exp\!\big(\tfrac{272{,}59-1054{,}10}{250}\big)}{1-\exp\!\big(\tfrac{-1054{,}10}{250}\big)}=0{,}970$$

Las dos variables reducen la desviación en un 74 %, y casi todo ese descenso lo aporta la zona.

In [ ]:
# ¿Cuánto aporta cada variable? Comparación de modelos anidados
m_nulo = smf.glm("casos ~ 1", data=DataE, family=familia_bn).fit()
m_zona = smf.glm("casos ~ C(zona, Treatment('AMAZ'))", data=DataE, family=familia_bn).fit()
m_anno = smf.glm("casos ~ anno", data=DataE, family=familia_bn).fit()

comparacion = pd.DataFrame({
    "Modelo": ["Solo intercepto", "Solo año", "Solo zona", "Zona + año"],
    "Desviación": [m_nulo.deviance, m_anno.deviance, m_zona.deviance, m_bn.deviance],
    "gl": [m_nulo.df_resid, m_anno.df_resid, m_zona.df_resid, m_bn.df_resid],
}).round(2)
comparacion["% de desviación explicada"] = (
    100*(1 - comparacion["Desviación"]/m_nulo.deviance)).round(1)
display(comparacion)

lr_anno = 2*(m_bn.llf - m_zona.llf)
lr_zona = 2*(m_bn.llf - m_anno.llf)
print(f"Aporte del año, con la zona ya en el modelo:  LR = {lr_anno:6.2f}, gl = 1, p = {st.chi2.sf(lr_anno,1):.3f}")
print(f"Aporte de la zona, con el año ya en el modelo: LR = {lr_zona:6.2f}, gl = 4, p = {st.chi2.sf(lr_zona,4):.3e}")

## 7. Por qué no una regresión lineal

Una pregunta razonable es por qué no usar el modelo lineal de siempre. Se ajusta aquí
justamente para mostrar dónde falla.

In [ ]:
m_ols = smf.ols("casos ~ C(anno) + C(zona, Treatment('AMAZ'))", data=DataE).fit()

etiquetas = ["(Intercept)", "anno|2021", "anno|2022", "anno|2023", "anno|2024",
             "zona|AND", "zona|CAR", "zona|ORIN", "zona|PACF"]
tols = pd.DataFrame({
    "coefficient": m_ols.params.values, "std.error": m_ols.bse.values,
    "t.value": m_ols.tvalues.values, "p.value": m_ols.pvalues.values,
}, index=etiquetas)
tols["signif"] = [estrellas(p) for p in tols["p.value"]]
print("Linear regression (OLS)")
print(tols.to_string(float_format=lambda v: f"{v:10.3f}"))
print(f"\nR-squared: {m_ols.rsquared:.3f},  Adjusted R-squared: {m_ols.rsquared_adj:.3f}")
print(f"F-statistic: {m_ols.fvalue:.3f} df({int(m_ols.df_model)},{int(m_ols.df_resid)}), p.value < 0.001")
print(f"Nr obs: {int(m_ols.nobs)}")

In [ ]:
# Predicción para 2020 en cada zona, con intervalo de confianza
nuevo = pd.DataFrame({"anno": 2020, "zona": ZONAS})
pred_ols = m_ols.get_prediction(nuevo).summary_frame(alpha=0.05)

pred = pd.DataFrame({
    "anno": 2020, "zona": ZONAS,
    "Prediction": pred_ols["mean"].values,
    "2.5%": pred_ols["mean_ci_lower"].values,
    "97.5%": pred_ols["mean_ci_upper"].values,
}).round(3)
pred["+/-"] = (pred["97.5%"] - pred["Prediction"]).round(3)
display(pred)

negativas = pred[pred["2.5%"] < 0]
print(f"Zonas con límite inferior negativo: {len(negativas)} de {len(pred)}")
for _, r in negativas.iterrows():
    print(f"   {r.zona}: intervalo de {r['2.5%']:.3f} a {r['97.5%']:.3f} casos")
print("\nUn conteo no puede ser negativo. El modelo lineal no lo sabe; el de conteo sí.")

### Figura 6 · Lo que predice cada modelo

Mismo año, mismas zonas, dos modelos. La zona sombreada es territorio imposible.

In [ ]:
# Predicción de la binomial negativa, con su intervalo
X = pd.DataFrame({"anno": 2020, "zona": ZONAS})
pb = m_bn.get_prediction(X).summary_frame(alpha=0.05)
pred_bn = pd.DataFrame({"zona": ZONAS, "est": pb["mean"].values,
                        "inf": pb["mean_ci_lower"].values, "sup": pb["mean_ci_upper"].values})

fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(ZONAS))
ax.axvspan(-10, 0, color="#E8552D", alpha=0.10, zorder=0)
ax.axvline(0, color=NARANJA, lw=1.6)
ax.text(-9.4, 1.55, "casos negativos:\nimposible", fontsize=10.5,
        color=NARANJA, va="center", style="italic")

ax.hlines(y+0.17, pred["2.5%"], pred["97.5%"], color="#9AA7B4", lw=3.2, zorder=2)
ax.scatter(pred["Prediction"], y+0.17, s=120, color="#4D5B6E", zorder=3,
           label="Regresión lineal")
ax.hlines(y-0.17, pred_bn["inf"], pred_bn["sup"], color="#F3B49C", lw=3.2, zorder=2)
ax.scatter(pred_bn["est"], y-0.17, s=120, color=NARANJA, zorder=3,
           label="Binomial negativa")

ax.set_yticks(y); ax.set_yticklabels(["Amazonía", "Andina", "Caribe", "Orinoquía", "Pacífico"],
                                     fontsize=12)
ax.set_xlim(-10, max(pred["97.5%"].max(), pred_bn["sup"].max())*1.09); ax.set_xlabel("Casos predichos para 2020 (intervalo del 95 %)")
ax.set_title("La recta predice casos negativos; el modelo de conteo no puede",
             loc="left", weight="bold", color=NAVY, fontsize=14, pad=12)
ax.legend(frameon=False, loc="lower right", fontsize=11)
panel_ggplot(ax)
fig.tight_layout()
fig.savefig("figuras/fig6_lineal_vs_conteo.png", bbox_inches="tight")
plt.show()

## 8. Resumen y exportación

In [ ]:
R = {
    "n_registros": int(len(DataE)), "n_casos": int(DataE.casos.sum()),
    "theta": THETA,
    "coeficientes": {k: float(v) for k, v in zip(nombres, m_bn.params.values)},
    "irr_zona": {NOMBRE_ZONA[k]: float(irr.loc[k, "IRR"]) for k in NOMBRE_ZONA},
    "p_anno": float(irr.loc["anno", "p"]),
    "desviacion_nula": float(m_bn.null_deviance), "desviacion_residual": float(m_bn.deviance),
    "AIC": float(AIC), "AICc": float(AICc), "BIC": float(BIC),
    "R2_nagelkerke": float(R2N), "RMSE": float(RMSE),
    "ols_r2": float(m_ols.rsquared), "ols_r2_ajustado": float(m_ols.rsquared_adj),
    "ols_zonas_con_ic_negativo": negativas.zona.tolist(),
}

resumen = pd.DataFrame([
    ("Casos analizados", f"{R['n_casos']:,}".replace(",", ".") + f" en {R['n_registros']} registros"),
    ("Zona Andina frente a la Amazonía", f"{R['irr_zona']['Andina']:.1f} × (p < 0,001)"),
    ("Caribe frente a la Amazonía", f"{R['irr_zona']['Caribe']:.1f} × (p < 0,001)"),
    ("Pacífico frente a la Amazonía", f"{R['irr_zona']['Pacífico']:.1f} × (p < 0,001)"),
    ("Orinoquía frente a la Amazonía", f"{R['irr_zona']['Orinoquía']:.1f} × (p < 0,001)"),
    ("Efecto del año", f"no significativo (p = {R['p_anno']:.3f})"),
    ("R² de Nagelkerke", f"{R['R2_nagelkerke']:.3f}"),
    ("Desviación", f"de {R['desviacion_nula']:.2f} a {R['desviacion_residual']:.2f}"),
    ("Regresión lineal: R²", f"{R['ols_r2']:.3f}"),
    ("Regresión lineal: zonas con intervalo negativo", ", ".join(R["ols_zonas_con_ic_negativo"])),
], columns=["Resultado", "Valor"])
display(resumen)

resumen.to_csv("resultados/resumen.csv", index=False, encoding="utf-8")
tabla.to_csv("resultados/coeficientes_binomial_negativa.csv", encoding="utf-8")
zonas_irr.to_csv("resultados/irr_por_zona.csv", encoding="utf-8")
tols.to_csv("resultados/coeficientes_regresion_lineal.csv", encoding="utf-8")
pred.to_csv("resultados/predicciones_lineal_2020.csv", index=False, encoding="utf-8")
json.dump(R, open("resultados/resultados.json", "w", encoding="utf-8"),
          indent=1, ensure_ascii=False)

print("\nArchivos guardados:")
for c in ["figuras", "resultados"]:
    for f in sorted(os.listdir(c)):
        print(f"   {c}/{f}")

### Lectura de los resultados

1. **La zona explica casi todo.** Las cuatro comparaciones contra la Amazonía son
   significativas y de magnitud grande: la Andina registra 25 veces más casos.
2. **El año no explica nada.** Su coeficiente no es significativo ni en el modelo de
   conteo (p = 0,352) ni en la regresión lineal, donde ningún año se separa de 2020.
   Entre 2020 y 2024 el patrón geográfico se mantuvo estable.
3. **El modelo de conteo era necesario.** La regresión lineal produce intervalos con
   casos negativos en dos de las cinco zonas. La binomial negativa respeta la naturaleza
   discreta y no negativa del dato.

**Una advertencia al leer estas cifras.** Se analizan conteos de notificación, no tasas de
incidencia. Sin denominadores poblacionales, la diferencia entre zonas refleja también
cuánta población infantil vive en cada una y qué capacidad de diagnóstico y notificación
tiene. Con las proyecciones de población del DANE, este mismo cuaderno permitiría calcular
tasas por 100.000 y razones de incidencia estandarizada.

---

### Referencias

- Venables, W. N. y Ripley, B. D. (2002). *Modern Applied Statistics with S* (4.ª ed.). Springer. (paquete `MASS`)
- Cameron, A. C. y Trivedi, P. K. (2013). *Regression analysis of count data* (2.ª ed.). Cambridge University Press.
- Lüdecke, D. et al. (2021). performance: An R package for assessment, comparison and testing of statistical models. *Journal of Open Source Software*, 6(60), 3139.
- R Core Team (2023). *R: A language and environment for statistical computing*.